# Notebook 01 — Problem statement & EDA (Checkpoints 1 & 2)

**Name:** Anthony Howell  
**Dataset:** Craigslist-style used vehicle listings (`vehicles` dataset) — loaded from your full `vehicles.csv`  
**Target for the whole capstone:** listing **price** (numeric dollars)

---



## Setup


In [1]:
import pandas as pd  #I'm doing this because pandas reads CSV files into DataFrames and supports cleaning and stats.
import numpy as np  #I'm doing this because numpy provides numeric helpers such as nan and vectorized operations.
import matplotlib.pyplot as plt  #I'm doing this because matplotlib draws plots for exploratory data analysis.
import seaborn as sns  #I'm doing this because seaborn adds convenient statistical plots on top of matplotlib.
import warnings  #I'm doing this because I need the warnings API to silence noisy library messages.

warnings.filterwarnings("ignore")  #I'm doing this because sklearn and matplotlib can flood the notebook with benign warnings.
pd.set_option("display.max_columns", None)  #I'm doing this because the vehicles table has many columns I want to see in full.
plt.style.use("seaborn-v0_8-whitegrid")  #I'm doing this because a light grid improves readability of distribution plots.
print("ok — libraries loaded")  #I'm doing this because a short confirmation proves imports completed successfully.


ok — libraries loaded


---

# CHECKPOINT 1

## 1) Problem statement

**What am I trying to predict?**  
The listed **price** of a used vehicle.

**Why do I care?**  
Used car listings are noisy. People lie, people typo, people “test the market” with fantasy prices. If you can ballpark a fair-ish price from a few structured fields (year, miles, drivetrain, etc.), you at least get a sanity check before you drive across town.

**What features do I think will matter (before I look)?**  
Year/age, odometer, manufacturer, fuel, transmission, drive, and body type. That’s mostly “how worn is it” + “what is it”.


## 2) Dataset description


This is basically the classic **Craigslist vehicles** dump people use for ML demos. It’s not perfect science data — it’s messy internet postings — but that’s also why it’s a good classroom dataset.

This notebook loads **your full** `vehicles.csv` from your Downloads capstone folder (not a small sample file) so every EDA cell reflects the same data you will model.


## 3) Load data + first peek


In [2]:
from pathlib import Path  #I'm doing this because pathlib builds cross-platform paths to your CSV and output folders.

PROJECT_ROOT = Path("/Users/jahbrae/Downloads/NEW")  #I'm doing this because all outputs from this NEW project land under this root.
RAW_PATH = Path("/Users/jahbrae/Downloads/INDIVIDUAL CAPSTONE PROJECT ") / "vehicles.csv"  #I'm doing this because this points to your full vehicles listing file.
df = pd.read_csv(RAW_PATH, low_memory=False)  #I'm doing this because low_memory=False avoids mixed-type column inference issues on large CSVs.
print("shape:", df.shape)  #I'm doing this because row and column counts summarize dataset size before deeper EDA.
df.head()  #I'm doing this because the first rows show real values for each column and catch obvious parsing problems.


shape: (426880, 26)


,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,https://worcester.craigslist.org/cto/d/west-br...,worcester / central MA,https://worcester.craigslist.org,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,https://greensboro.craigslist.org/cto/d/trinit...,greensboro,https://greensboro.craigslist.org,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN


In [3]:
df.dtypes  #I'm doing this because dtype per column tells me which fields are numeric versus text before coercion.


id                int64
url              object
region           object
region_url       object
price             int64
year            float64
manufacturer     object
model            object
condition        object
cylinders        object
fuel             object
odometer        float64
title_status     object
transmission     object
VIN              object
drive            object
size             object
type             object
paint_color      object
image_url        object
description      object
county          float64
state            object
lat             float64
long            float64
posting_date     object
dtype: object

In [4]:
TARGET = "price"  #I'm doing this because price is the regression target for the whole capstone.
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")  #I'm doing this because prices may appear as strings and must become floats for stats.
print(df[TARGET].describe())  #I'm doing this because summary stats reveal scale, missingness, and extreme tails on price.


count    4.268800e+05
mean     7.519903e+04
std      1.218228e+07
min      0.000000e+00
25%      5.900000e+03
50%      1.395000e+04
75%      2.648575e+04
max      3.736929e+09
Name: price, dtype: float64


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))  #I'm doing this because an 8x4 figure gives a readable single histogram.
sns.histplot(df[TARGET].dropna(), bins=60, ax=ax)  #I'm doing this because 60 bins show skew while dropna avoids NaN errors.
ax.set_title("Price is heavy-tailed (welcome to real life)")  #I'm doing this because the title states the main EDA takeaway.
plt.show()  #I'm doing this because show renders the figure inline in the notebook.


### Checkpoint 1 reflection

Nothing shocking yet: shape is reasonable, types are mostly strings because the internet is strings, and price has a long right tail. That tail is exactly why later I clip insane prices instead of pretending every row is real.


---

# CHECKPOINT 2

## 4) Deeper EDA (relationships + correlations)

I’m not gonna plot every column — a lot of them are URLs / text blobs that aren’t helpful for a clean modeling table.


In [6]:
for col in ["year", "odometer"]:  #I'm doing this because year and mileage are core numeric predictors for vehicle price.
    if col in df.columns:  #I'm doing this because missing column names would error on slightly different schema versions.
        df[col] = pd.to_numeric(df[col], errors="coerce")  #I'm doing this because these fields often load as strings from CSV.

eda = df[["price", "year", "odometer"]].dropna()  #I'm doing this because pairplots need complete rows across these three variables.
sample_n = min(2000, len(eda))  #I'm doing this because plotting millions of points is slow and 2000 is enough to see structure.
sns.pairplot(eda.sample(sample_n, random_state=42), corner=True)  #I'm doing this because corner=True avoids duplicate upper panels.
plt.show()  #I'm doing this because it displays the pairplot matrix in the notebook output.


In [7]:
corr = eda.corr(numeric_only=True)  #I'm doing this because Pearson correlation quantifies linear relationships among numerics.
sns.heatmap(corr, annot=True, cmap="vlag", center=0)  #I'm doing this because a diverging colormap highlights positive versus negative correlation.
plt.title("Correlation heatmap (small but useful)")  #I'm doing this because the title frames what the heatmap represents.
plt.show()  #I'm doing this because it renders the heatmap figure.
print(corr)  #I'm doing this because printed numbers complement the color scale for exact correlation values.


             price      year  odometer
price     1.000000 -0.004940  0.010032
year     -0.004940  1.000000 -0.157215
odometer  0.010032 -0.157215  1.000000


## 5) Cleaning (the part nobody posts on Instagram)


In [8]:
DROP = [  #I'm doing this because this list names columns to remove for a lean modeling table without URL or NLP fields.
    "id", "url", "region_url", "image_url", "description", "VIN", "lat", "long",  #I'm doing this because identifiers and blobs are not inputs for the tabular models.
    "posting_date", "county", "region", "model", "state", "paint_color", "title_status", "size",  #I'm doing this because these are high-cardinality or weakly structured for this assignment scope.
]  #I'm doing this because closing the list completes the DROP definition.
clean = df.drop(columns=[c for c in DROP if c in df.columns], errors="ignore").copy()  #I'm doing this because copy avoids SettingWithCopy warnings when I mutate clean.

clean["price"] = pd.to_numeric(clean["price"], errors="coerce")  #I'm doing this because downstream filters require numeric price.
clean["year"] = pd.to_numeric(clean["year"], errors="coerce")  #I'm doing this because model year must be numeric for range filters.
clean["odometer"] = pd.to_numeric(clean["odometer"], errors="coerce")  #I'm doing this because mileage must be numeric for filters and modeling.

clean = clean[(clean["price"] >= 500) & (clean["price"] <= 125_000)]  #I'm doing this because extreme tails often reflect scams or bad entries.
clean = clean[(clean["year"] >= 1990) & (clean["year"] <= 2022)]  #I'm doing this because impossible years break age features and dominate plots.
clean = clean[(clean["odometer"] >= 0) & (clean["odometer"] <= 400_000)]  #I'm doing this because impossible mileage values distort distance-based models.

for col in ["manufacturer", "fuel", "transmission", "drive", "type", "condition", "cylinders"]:  #I'm doing this because categoricals need consistent string cleanup before one-hot encoding.
    if col in clean.columns:  #I'm doing this because optional columns may be absent in some exports.
        clean[col] = clean[col].astype(str).str.strip().replace({"nan": np.nan})  #I'm doing this because literal 'nan' strings should become real missing values.

clean = clean.dropna(subset=["price", "year", "odometer", "manufacturer", "fuel", "transmission", "drive", "type"])  #I'm doing this because models need complete rows across all chosen predictors.

TOP_N = 12  #I'm doing this because keeping the top twelve manufacturers limits one-hot dimensionality.
top_m = clean["manufacturer"].value_counts().nlargest(TOP_N).index  #I'm doing this because this captures the most frequent brand labels.
clean["manufacturer"] = np.where(clean["manufacturer"].isin(top_m), clean["manufacturer"], "other")  #I'm doing this because rare brands are grouped to stabilize estimates.

print("clean shape:", clean.shape)  #I'm doing this because the final row count confirms how many listings survived cleaning.


clean shape: (221082, 10)


### Cleaning reflection

Yeah, I’m intentionally deleting a bunch of columns. Not because they’re “bad,” but because for *this* assignment I’m not doing NLP on the description field. I’m building a small app with a handful of inputs. URLs and VIN aren’t helping the user experience.

The numeric clipping is me drawing a line in the sand: ultra-cheap and ultra-expensive tails are often data quality issues or weird edge cases, and they wreck models for normal listings.


## 6) Feature engineering


In [9]:
age_base = int(clean["year"].max() + 1)  #I'm doing this because anchoring age off max year plus one defines interpretable vehicle age in the sample.
clean["vehicle_age"] = age_base - clean["year"]  #I'm doing this because subtracting year from the anchor yields years since manufacture.

q1, q2 = clean["price"].quantile([1 / 3, 2 / 3])  #I'm doing this because tertile cutoffs create three roughly equal-frequency price tiers.


def tier(p):  #I'm doing this because a small function maps each numeric price to a discrete tier label.
    if p <= q1:  #I'm doing this because prices at or below the first tertile belong to the budget bucket.
        return "budget"  #I'm doing this because this label names the lowest price third.
    if p <= q2:  #I'm doing this because prices between tertiles one and two define the mid bucket.
        return "mid"  #I'm doing this because this label names the middle price third.
    return "premium"  #I'm doing this because prices above the second tertile define the premium bucket.


clean["price_tier"] = clean["price"].map(tier)  #I'm doing this because the tier column is the classification target for notebook 03.
print("tertile cutoffs:", float(q1), float(q2))  #I'm doing this because printing cutoffs documents the exact boundaries used for bins.
clean[["price", "price_tier"]].head()  #I'm doing this because a quick peek verifies tiers align with prices monotonically.


tertile cutoffs: 10600.0 23495.0


,price,price_tier
31,15000,mid
32,27990,premium
33,34590,premium
34,35000,premium
35,29990,premium


### Feature engineering reflection

`vehicle_age` is the same idea as “how old is the car” but anchored off the newest year in the cleaned sample so it lines up with what I ship in the app.

For classification, tertiles are a boring but defensible default: three buckets, roughly equal support, easy to explain to a professor (and to Abishek on Slack).


## 7) Save processed data


In [10]:
out = PROJECT_ROOT / "data" / "processed" / "cleaned_data.csv"  #I'm doing this because processed data should live under NEW for notebooks 02 and 03.
out.parent.mkdir(parents=True, exist_ok=True)  #I'm doing this because the processed folder may not exist on first run.
clean.to_csv(out, index=False)  #I'm doing this because CSV without the index avoids an extra unnamed column in later reads.
print("wrote", out.resolve())  #I'm doing this because the absolute path confirms where the cleaned file was written.


wrote /Users/jahbrae/Downloads/NEW/data/processed/cleaned_data.csv
